<a href="https://colab.research.google.com/github/Sencer1/pytorch/blob/main/deneme333.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install jsbsim
!pip install gymnasium
import jsbsim
import os
import numpy as np
import gymnasium as gym

In [47]:
!python --version

Python 3.13.15


In [49]:
!sudo apt-get update -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [51]:
!sudo apt-get install python3.11 python3.11-dev python3.11-distutils -y -qq

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 9.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package libpython3.11-minimal:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../0-libpython3.11-minimal_3.11.16-1+noble1_amd64.deb ...
Unpacking libpython3.11-minimal:amd64 (3.11.16-1+noble1) ...
Selecting previously unselected package python3.11-minimal.
Preparing to unpack .../1-python3.11-minimal_3.11.16-1+noble1_amd64.deb ...
Unpacking python3.11-minimal (3.11.16-1+noble1) ...
Selecting previously unselected package libpython3.11-stdlib:amd64.
Preparin

In [52]:
!python3.11 --version

Python 3.11.16


In [9]:
class FlightEnv(gym.Env):

  def __init__(self):
    super().__init__()

    self.fdm = jsbsim.FGFDMExec(jsbsim.get_default_root_dir())
    self.fdm.load_model("c172p")

    self.action_space = gym.spaces.Box(
        low=np.array([-1, -1, -1, 0]), # elevator, aileron, rudder, throttle
        high=np.array([1, 1, 1, 1]),
        dtype=np.float32
    )

    self.observation_space = gym.spaces.Box(
        low=-np.inf,
        high=np.inf,
        shape=(6,), # altitude, velocity, pitch, roll, yaw, vertical_speed
        dtype=np.float32
    )

  # reset episode başı için başlangıç durumlarını getirmek için
  def reset(self, seed=None, options=None):
    super().reset(seed=seed)

    self.fdm =jsbsim.FGFDMExec(jsbsim.get_default_root_dir())
    self.fdm.load_model("c172p")

    self.fdm["ic/h-sl-ft"] = 5000 # yükselik
    self.fdm["ic/vc-kts"] = 120 # başlangıç hızı
    self.fdm["ic/terrain-elevation-ft"] = 0 #zemin yüksekliği

    self.fdm.run_ic()

    observation = self._get_observation()
    info = {}

    return observation, info

  # step(action) ileri götürmek için state'i
  # istediğimiz staleri ppo ya vermek için burası
  def _get_observation(self):

    altitude = self.fdm["position/h-sl-ft"]
    velocity = self.fdm["velocities/vtrue-kts"]
    pitch = self.fdm["attitude/theta-rad"]
    roll = self.fdm["attitude/phi-rad"]
    yaw = self.fdm["attitude/psi-rad"]
    vertical_speed = self.fdm["velocities/v-down-fps"]

    observation = np.array([
        altitude,
        velocity,
        pitch,
        roll,
        yaw,
        vertical_speed
    ])

    return observation

In [10]:
env = FlightEnv()

observation, info = env.reset()

print("Observation: ", observation)



     JSBSim Flight Dynamics Model v1.3.1 May 17 2026 14:26:04
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

Reading Aircraft Configuration File: c172
                            Version: 2.0


This aircraft model is a BETA release!!!

This aircraft model probably will not fly as expected.

Use this model for development purposes ONLY!!!

  Description:   Cessna C-172
  Model Author:  Unknown
  Creation Date: 2002-01-01
  Version:       $Id: c172p.xml,v 1.29 2013/10/25 15:32:49 jentron Exp $

  Aircraft Metrics:
    WingArea: 174.000000
    WingSpan: 35.800000
    Incidence: 0.000000
    Chord: 4.900000
    H. Tail Area: 21.900000
    H. Tail Arm: 15.700000
    V. Tail Area: 16.500000
    V. Tail Arm: 0.000000
    Eyepoint (x, y, z): 37.000000 , 0.000000 , 48.000000
    Ref Pt (x, y, z): 43.200000 , 0.000000 , 59.400000
    Visual Ref Pt (x, y, z): 42.600000 , 0.000000 , 38.500000

  Mass and Balance:
    baseIxx: 948.000000 slug-ft2
    baseIyy: 1346.000000 slug-ft2
   

In [14]:
"""
step()
ppo action step(action) action ı jsbsim e gönderir jsbsim çlaışır yeni observation reward yeni döngü
"""

In [20]:
class FlightEnv(gym.Env):

  def __init__(self):
    super().__init__()

    self.fdm = jsbsim.FGFDMExec(jsbsim.get_default_root_dir())
    self.fdm.load_model("c172p")

    self.action_space = gym.spaces.Box(
        low=np.array([-1, -1, -1, 0]), # elevator, aileron, rudder, throttle
        high=np.array([1, 1, 1, 1]),
        dtype=np.float32
    )

    self.observation_space = gym.spaces.Box(
        low=-np.inf,
        high=np.inf,
        shape=(6,), # altitude, velocity, pitch, roll, yaw, vertical_speed
        dtype=np.float32
    )

  # reset episode başı için başlangıç durumlarını getirmek için
  def reset(self, seed=None, options=None):
    super().reset(seed=seed)

    self.fdm =jsbsim.FGFDMExec(jsbsim.get_default_root_dir())
    self.fdm.load_model("c172p")

    self.fdm["ic/h-sl-ft"] = 5000 # yükselik
    self.fdm["ic/vc-kts"] = 120 # başlangıç hızı
    self.fdm["ic/terrain-elevation-ft"] = 0 #zemin yüksekliği

    self.fdm.run_ic()

    observation = self._get_observation()
    info = {}

    return observation, info

  # step(action) ileri götürmek için state'i
  # istediğimiz staleri ppo ya vermek için burası
  def _get_observation(self):

    altitude = self.fdm["position/h-sl-ft"]
    velocity = self.fdm["velocities/vtrue-kts"]
    pitch = self.fdm["attitude/theta-rad"]
    roll = self.fdm["attitude/phi-rad"]
    yaw = self.fdm["attitude/psi-rad"]
    vertical_speed = self.fdm["velocities/v-down-fps"]

    observation = np.array([
        altitude,
        velocity,
        pitch,
        roll,
        yaw,
        vertical_speed
    ])

    return observation

  def step(self, action):
    elevator = action[0]
    aileron = action[1]
    rudder = action[2]
    throttle = action[3]

    self.fdm["fcs/elevator-cmd-norm"] = float(elevator)
    self.fdm["fcs/aileron-cdm-norm"] = float(aileron)
    self.fdm["fcs/rudder-cmd-norm"] = float(rudder)
    self.fdm["fcs/throtlle-cmd-norm"] = float(throttle)

    # bir adım ileri götürmek için
    self.fdm.run()

    observation = self._get_observation()

    reward = 0.0

    terminated = False
    truncated = False

    info = {}

    return observation, reward, terminated, truncated, info


In [21]:
env = FlightEnv()

env.reset()

action = np.array([0.2, 0.0, 0.0, 0.8], dtype=np.float32)

observation, reward, terminated, truncated, info = env.step(action)

print("observation:", observation)
print("reward", reward)
print("termianted", terminated)
print("truncated", truncated)



     JSBSim Flight Dynamics Model v1.3.1 May 17 2026 14:26:04
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

Reading Aircraft Configuration File: c172
                            Version: 2.0


This aircraft model is a BETA release!!!

This aircraft model probably will not fly as expected.

Use this model for development purposes ONLY!!!

  Description:   Cessna C-172
  Model Author:  Unknown
  Creation Date: 2002-01-01
  Version:       $Id: c172p.xml,v 1.29 2013/10/25 15:32:49 jentron Exp $

  Aircraft Metrics:
    WingArea: 174.000000
    WingSpan: 35.800000
    Incidence: 0.000000
    Chord: 4.900000
    H. Tail Area: 21.900000
    H. Tail Arm: 15.700000
    V. Tail Area: 16.500000
    V. Tail Arm: 0.000000
    Eyepoint (x, y, z): 37.000000 , 0.000000 , 48.000000
    Ref Pt (x, y, z): 43.200000 , 0.000000 , 59.400000
    Visual Ref Pt (x, y, z): 42.600000 , 0.000000 , 38.500000

  Mass and Balance:
    baseIxx: 948.000000 slug-ft2
    baseIyy: 1346.000000 slug-ft2
   

In [23]:
class FlightEnv(gym.Env):

  def __init__(self):
    super().__init__()

    self.fdm = jsbsim.FGFDMExec(jsbsim.get_default_root_dir())
    self.fdm.load_model("c172p")

    self.action_space = gym.spaces.Box(
        low=np.array([-1, -1, -1, 0]), # elevator, aileron, rudder, throttle
        high=np.array([1, 1, 1, 1]),
        dtype=np.float32
    )

    self.observation_space = gym.spaces.Box(
        low=-np.inf,
        high=np.inf,
        shape=(6,), # altitude, velocity, pitch, roll, yaw, vertical_speed
        dtype=np.float32
    )

  # reset episode başı için başlangıç durumlarını getirmek için
  def reset(self, seed=None, options=None):
    super().reset(seed=seed)

    self.fdm =jsbsim.FGFDMExec(jsbsim.get_default_root_dir())
    self.fdm.load_model("c172p")

    self.fdm["ic/h-sl-ft"] = 5000 # yükselik
    self.fdm["ic/vc-kts"] = 120 # başlangıç hızı
    self.fdm["ic/terrain-elevation-ft"] = 0 #zemin yüksekliği

    self.fdm.run_ic()

    observation = self._get_observation()
    info = {}

    return observation, info

  # step(action) ileri götürmek için state'i
  # istediğimiz staleri ppo ya vermek için burası
  def _get_observation(self):

    altitude = self.fdm["position/h-sl-ft"]
    velocity = self.fdm["velocities/vtrue-kts"]
    pitch = self.fdm["attitude/theta-rad"]
    roll = self.fdm["attitude/phi-rad"]
    yaw = self.fdm["attitude/psi-rad"]
    vertical_speed = self.fdm["velocities/v-down-fps"]

    observation = np.array([
        altitude,
        velocity,
        pitch,
        roll,
        yaw,
        vertical_speed
    ])

    return observation

  def step(self, action):
    elevator = action[0]
    aileron = action[1]
    rudder = action[2]
    throttle = action[3]

    self.fdm["fcs/elevator-cmd-norm"] = float(elevator)
    self.fdm["fcs/aileron-cdm-norm"] = float(aileron)
    self.fdm["fcs/rudder-cmd-norm"] = float(rudder)
    self.fdm["fcs/throtlle-cmd-norm"] = float(throttle)

    # bir adım ileri götürmek için
    self.fdm.run()

    observation = self._get_observation()

    target_pitch = 0.0
    pitch = self.fdm["attitude/theta-rad"]
    error = pitch - target_pitch
    reward = -abs(error)

    terminated = False
    truncated = False

    info = {}

    return observation, reward, terminated, truncated, info


In [54]:
env.reset()
action1 = np.array([0.0, 0.0, 0.0, 0.5], dtype=np.float32)
for i in range(1000):
  observation1, reward, terminated, truncated, info = env.step(action1)
env.reset()
action2 = np.array([0.5, 0.0, 0.0, 0.5], dtype=np.float32)
for i in range(1000):
  observation2, reward, terminated, truncated, info = env.step(action2)



     JSBSim Flight Dynamics Model v1.3.1 May 17 2026 14:26:04
            [JSBSim-ML v2.0]

JSBSim startup beginning ...

Reading Aircraft Configuration File: c172
                            Version: 2.0


This aircraft model is a BETA release!!!

This aircraft model probably will not fly as expected.

Use this model for development purposes ONLY!!!

  Description:   Cessna C-172
  Model Author:  Unknown
  Creation Date: 2002-01-01
  Version:       $Id: c172p.xml,v 1.29 2013/10/25 15:32:49 jentron Exp $

  Aircraft Metrics:
    WingArea: 174.000000
    WingSpan: 35.800000
    Incidence: 0.000000
    Chord: 4.900000
    H. Tail Area: 21.900000
    H. Tail Arm: 15.700000
    V. Tail Area: 16.500000
    V. Tail Arm: 0.000000
    Eyepoint (x, y, z): 37.000000 , 0.000000 , 48.000000
    Ref Pt (x, y, z): 43.200000 , 0.000000 , 59.400000
    Visual Ref Pt (x, y, z): 42.600000 , 0.000000 , 38.500000

  Mass and Balance:
    baseIxx: 948.000000 slug-ft2
    baseIyy: 1346.000000 slug-ft2
   

In [55]:
print(observation1)
print("-" * 60)
print(observation2)

[ 5.46097988e+03  4.81185475e+01  4.23105992e-01 -2.65335006e-01
  6.04925372e+00 -2.55861358e+01]
------------------------------------------------------------
[ 4.24604696e+03  1.58057684e+02 -8.05245084e-01  1.23056578e-02
  8.73712754e-03  1.81449429e+02]


In [ ]:
class FlightEnv(gym.Env):

  def __init__(self):
    super().__init__()

    self.fdm = jsbsim.FGFDMExec(jsbsim.get_default_root_dir())
    self.fdm.load_model("c172p")

    self.action_space = gym.spaces.Box(
        low=np.array([-1, -1, -1, 0]), # elevator, aileron, rudder, throttle
        high=np.array([1, 1, 1, 1]),
        dtype=np.float32
    )

    self.observation_space = gym.spaces.Box(
        low=-np.inf,
        high=np.inf,
        shape=(6,), # altitude, velocity, pitch, roll, yaw, vertical_speed
        dtype=np.float32
    )

    self.step_count = 0
    self.max_steps = 500

  # reset episode başı için başlangıç durumlarını getirmek için
  def reset(self, seed=None, options=None):
    super().reset(seed=seed)

    self.step_count = 0

    self.fdm =jsbsim.FGFDMExec(jsbsim.get_default_root_dir())
    self.fdm.load_model("c172p")

    self.fdm["ic/h-sl-ft"] = 5000 # yükselik
    self.fdm["ic/vc-kts"] = 120 # başlangıç hızı
    self.fdm["ic/terrain-elevation-ft"] = 0 #zemin yüksekliği

    self.fdm.run_ic()

    observation = self._get_observation()
    info = {}

    return observation, info

  # step(action) ileri götürmek için state'i
  # istediğimiz staleri ppo ya vermek için burası
  def _get_observation(self):

    altitude = self.fdm["position/h-sl-ft"]
    velocity = self.fdm["velocities/vtrue-kts"]
    pitch = self.fdm["attitude/theta-rad"]
    roll = self.fdm["attitude/phi-rad"]
    yaw = self.fdm["attitude/psi-rad"]
    vertical_speed = self.fdm["velocities/v-down-fps"]

    observation = np.array([
        altitude,
        velocity,
        pitch,
        roll,
        yaw,
        vertical_speed
    ])

    return observation

  def step(self, action):
    elevator = action[0]
    aileron = action[1]
    rudder = action[2]
    throttle = action[3]

    self.fdm["fcs/elevator-cmd-norm"] = float(elevator)
    self.fdm["fcs/aileron-cdm-norm"] = float(aileron)
    self.fdm["fcs/rudder-cmd-norm"] = float(rudder)
    self.fdm["fcs/throtlle-cmd-norm"] = float(throttle)

    # bir adım ileri götürmek için
    self.fdm.run()

    self.step_count += 1

    observation = self._get_observation()

    target_pitch = 0.0
    pitch = self.fdm["attitude/theta-rad"]
    error = pitch - target_pitch
    reward = -abs(error)

    terminated = False
    truncated = self.step_count >= self.max_steps

    info = {}

    return observation, reward, terminated, truncated, info
